In [0]:
%pip install mlflow openai --quiet
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import mlflow
import pandas as pd
from openai import OpenAI

# Databricks built-in LLM - no API key needed!
client = OpenAI(
    base_url="https://dbc-820598a7-e451.cloud.databricks.com/serving-endpoints",
    api_key=dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
)

# Load our cleaned data
df = spark.table("default.ghana_facilities_clean").toPandas()
print("Loaded:", len(df), "facilities")

# Test the LLM connection
response = client.chat.completions.create(
    model="databricks-meta-llama-3-3-70b-instruct",
    messages=[{"role": "user", "content": "Say 'LLM connected!' and nothing else."}],
    max_tokens=20
)
print(response.choices[0].message.content)

Loaded: 995 facilities
LLM connected!


In [0]:
import json
import mlflow

mlflow.set_experiment("/IDP_Agent")

SYSTEM_PROMPT = """You are a medical facility information extractor.
Extract structured facts from the given facility information.

Return ONLY a valid JSON object with exactly these three keys:
{
  "procedure": ["list of clinical procedures performed here"],
  "equipment": ["list of physical medical devices and infrastructure"],
  "capability": ["list of medical capabilities and care levels"]
}

Rules:
- Each item must be a clear declarative statement
- Return empty lists [] if no relevant facts found
- Do NOT include addresses, contact info, or pricing
- Only extract facts supported by the provided content"""

def run_idp_agent(row):
    # Build context from all available info
    context = f"""
Facility: {row.get('name', 'Unknown')}
Type: {row.get('facilityTypeId', 'Unknown')}
City: {row.get('address_city', 'Unknown')}
Region: {row.get('address_stateOrRegion', 'Unknown')}
Description: {row.get('description', '')}
Existing procedures: {row.get('procedure', '')}
Existing equipment: {row.get('equipment', '')}
Existing capabilities: {row.get('capability', '')}
Specialties: {row.get('specialties', '')}
"""

    with mlflow.start_run(run_name=f"IDP_{row.get('unique_id', 'unknown')}", nested=True):
        mlflow.log_param("facility", row.get('name', 'Unknown'))
        mlflow.log_param("city", row.get('address_city', 'Unknown'))

        response = client.chat.completions.create(
            model="databricks-meta-llama-3-3-70b-instruct",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": context}
            ],
            max_tokens=500
        )

        raw = response.choices[0].message.content.strip()

        try:
            # Clean response and parse JSON
            clean = raw.replace("```json", "").replace("```", "").strip()
            parsed = json.loads(clean)
        except:
            parsed = {"procedure": [], "equipment": [], "capability": []}

        mlflow.log_text(raw, "llm_response.txt")
        mlflow.log_dict(parsed, "extracted_facts.json")

        return parsed

# Test on 3 facilities first
print("Testing IDP agent on 3 facilities...\n")
sample = df.head(3)

with mlflow.start_run(run_name="IDP_Agent_Test"):
    for _, row in sample.iterrows():
        result = run_idp_agent(row)
        print(f"Facility: {row['name']}")
        print(f"  Procedures:   {result['procedure']}")
        print(f"  Equipment:    {result['equipment']}")
        print(f"  Capabilities: {result['capability']}")
        print()

Testing IDP agent on 3 facilities...

Facility: 109/No 1 Bekwai Rd (Near Mexico Hotel) Takoradi, Ghana
  Procedures:   ['HIV Testing and Counseling', 'Behavior Change Communication', 'Community Outreach', 'Hospice / Home Based Care']
  Equipment:    []
  Capabilities: ['Deliver prevention services', 'Deliver care and support services', 'Stigma Reduction', 'Global health services', 'Public health services', 'HIV and AIDS care', 'PMTCT']

Facility: 1st Foundation Clinic
  Procedures:   []
  Equipment:    []
  Capabilities: ['Provides internal medicine specialty care']

Facility: 1st Foundation Clinic
  Procedures:   []
  Equipment:    []
  Capabilities: ['internal medicine']



In [0]:
import time

# Run IDP on all facilities
print("Running IDP Agent on all 995 facilities...")
print("This will take a few minutes...\n")

results = []

with mlflow.start_run(run_name="IDP_Agent_Full_Run"):
    for i, (_, row) in enumerate(df.iterrows()):
        try:
            result = run_idp_agent(row)
            results.append({
                "unique_id": row.get("unique_id", ""),
                "name": row.get("name", ""),
                "address_city": row.get("address_city", ""),
                "address_stateOrRegion": row.get("address_stateOrRegion", ""),
                "facilityTypeId": row.get("facilityTypeId", ""),
                "idp_procedures": json.dumps(result["procedure"]),
                "idp_equipment": json.dumps(result["equipment"]),
                "idp_capabilities": json.dumps(result["capability"]),
                "has_procedures": len(result["procedure"]) > 0,
                "has_equipment": len(result["equipment"]) > 0,
                "has_capabilities": len(result["capability"]) > 0,
            })
        except Exception as e:
            print(f"  Error on {row.get('name', 'unknown')}: {e}")
            results.append({
                "unique_id": row.get("unique_id", ""),
                "name": row.get("name", ""),
                "address_city": row.get("address_city", ""),
                "address_stateOrRegion": row.get("address_stateOrRegion", ""),
                "facilityTypeId": row.get("facilityTypeId", ""),
                "idp_procedures": "[]",
                "idp_equipment": "[]",
                "idp_capabilities": "[]",
                "has_procedures": False,
                "has_equipment": False,
                "has_capabilities": False,
            })

        # Progress update every 50
        if (i + 1) % 50 == 0:
            print(f"  Processed {i+1}/995 facilities...")
        
        time.sleep(0.3)  # avoid rate limiting

# Save results
results_df = pd.DataFrame(results)
spark.createDataFrame(results_df).write.mode("overwrite").saveAsTable("default.ghana_idp_results")

print(f"\nDone! Processed {len(results)} facilities")
print(f"Has procedures: {results_df['has_procedures'].sum()}")
print(f"Has equipment: {results_df['has_equipment'].sum()}")
print(f"Has capabilities: {results_df['has_capabilities'].sum()}")
print("Saved to: default.ghana_idp_results")

Running IDP Agent on all 995 facilities...
This will take a few minutes...

  Processed 50/995 facilities...
  Processed 100/995 facilities...
  Processed 150/995 facilities...
  Processed 200/995 facilities...
  Processed 250/995 facilities...
  Processed 300/995 facilities...
  Processed 350/995 facilities...
  Processed 400/995 facilities...
  Processed 450/995 facilities...
  Processed 500/995 facilities...
  Processed 550/995 facilities...
  Processed 600/995 facilities...
  Processed 650/995 facilities...
  Processed 700/995 facilities...
  Processed 750/995 facilities...
  Processed 800/995 facilities...
  Processed 850/995 facilities...
  Processed 900/995 facilities...
  Processed 950/995 facilities...

Done! Processed 995 facilities
Has procedures: 277
Has equipment: 108
Has capabilities: 969
Saved to: default.ghana_idp_results
